# Assignment: Build a Multi-Agent Travel Planner

**Course topic:** multi-agent systems with **LangGraph** + local **Ollama** models
**Estimated time:** 3–4 hours  

> Fill in every cell marked **TODO**. Each task has a self-check cell: run it to see ✅ / ❌.

## The problem
Build a **Travel Planner** that, given a country, trip length, start date, daily budget and interests, recommends the **best city** based on **live weather** and **cost**, and writes a **day-by-day itinerary** that adapts to the forecast (indoor activities on rainy days).

Example request:
```python
{"country": "Japan", "days": 3, "start_in_days": 2, "budget_per_day_usd": 140,
  "interests": ["culture", "food"], "home_currency": "INR"}
```

## Architecture you will build

```
                     ┌──▶ weather_analyst ─┐
 START ─▶ scout ─────┤                     ├──▶ decider ─▶ planner ─▶ END
                     └──▶ budget_analyst ──┘
```

| Node | Type | Job |
|---|---|---|
| `scout` | **LLM agent** (structured output) | Shortlist up to 3 cities matching the traveller's interests |
| `weather_analyst` | tools + code *(given as an example)* | Fetch live forecast per city and score it |
| `budget_analyst` | tools + code | Compute trip cost, check budget, convert to home currency |
| `decider` | plain code | Combine scores → pick the best city |
| `planner` | **LLM agent** | Write the itinerary using forecast + the city's sights |

**Patterns practised:** structured output, tool use, parallel fan-out / fan-in, shared state, grounding an LLM in data to avoid hallucination.

## Tasks & marks
| # | Task | Marks |
|---|---|---|
| 1 | `score_weather` — turn a forecast into a 0–100 comfort score | 10 |
| 2 | `scout_node` — LLM shortlists cities (structured output + validation) | 15 |
| 3 | `budget_node` — cost, budget check, currency conversion | 15 |
| 4 | `decider_node` — rank and choose the best city | 15 |
| 5 | `planner_node` — weather-aware itinerary, grounded in given sights | 15 |
| 6 | Wire the LangGraph and run end-to-end | 15 |
| 7 | Reflection questions | 15 |
| ★ | Stretch goals | +10 |

## Agent profiles — what each node is trying to achieve

**1. Scout** *(LLM agent)*
- **Goal:** narrow a whole country down to up to 3 cities that fit the traveller's interests (culture, beach, food, ...).
- **Input:** the country and interests, plus every dataset city in that country with its tags.
- **Tools:** none; it uses the LLM's structured output and your validation code.
- **Output:** a shortlist of real city names (never invented ones).

**2. Weather Analyst** *(tools + code)*
- **Goal:** find out how pleasant each shortlisted city will be during the trip dates.
- **Input:** the shortlist, trip start and trip length. **Tools:** live Open-Meteo forecast and your `score_weather` function.
- **Output:** per-city daily forecast and a 0-100 comfort score.

**3. Budget Analyst** *(tools + code)*
- **Goal:** find out which shortlisted cities the traveller can actually afford.
- **Input:** the shortlist, trip length and daily budget. **Tools:** the city cost table and a live exchange rate (Frankfurter).
- **Output:** per-city total cost in USD and in the traveller's home currency, and whether it fits the budget.

**4. Decider** *(plain code, no LLM)*
- **Goal:** combine weather and cost into one decision. **Input:** weather scores and budget results from the two analysts above.
- **Tools:** none; it is a weighted score (60% weather, 40% affordability) with an over-budget fallback.
- **Output:** a ranking and the single best city. Deterministic logic is cheaper and more reliable than an LLM here.

**5. Itinerary Planner** *(LLM agent)*
- **Goal:** turn the chosen city into a practical day-by-day plan that adapts to the forecast.
- **Input:** the best city, its sights (indoor or outdoor) and each day's weather. **Tools:** none; it relies on a carefully grounded prompt.
- **Output:** Day 1 to Day N with morning, afternoon and evening plans, using only the listed sights and indoor ones on rainy days.

## 0. Setup

In [ ]:
%pip install -q langgraph langchain-ollama langchain-core pydantic requests

Start Ollama and pull a model:
```bash
ollama pull llama3.2
```

In [ ]:
import datetime, requests
from typing import Optional, TypedDict, Annotated
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

MODEL = "llama3.2"
llm = ChatOllama(model=MODEL, temperature=0)
print(llm.invoke("Reply with one word: ready?").content)

## 1. Given: data and tools (do not modify)

* `CITIES` — a small offline dataset: coordinates, **rough** mid-range daily cost per person in USD (hotel + food + local transport; illustrative only), interest tags, and sights flagged `"in"` (indoor) or `"out"` (outdoor).
* `get_forecast` — **live** daily forecast from Open-Meteo (free, no key). Works for trips starting within the next ~10 days.
* `get_fx` — **live** exchange rate from Frankfurter (free, no key).

In [ ]:
CITIES = {
 # ---- India
 "Jaipur":    dict(country="India", lat=26.91, lon=75.79, daily_usd=55,  tags=["culture","history","shopping","food"],
                   sights=[("Amber Fort","out"),("City Palace","in"),("Hawa Mahal","out"),("Jantar Mantar","out"),("Albert Hall Museum","in")]),
 "Goa":       dict(country="India", lat=15.50, lon=73.83, daily_usd=60,  tags=["beach","nightlife","relaxation","food"],
                   sights=[("Baga Beach","out"),("Basilica of Bom Jesus","in"),("Fort Aguada","out"),("Anjuna Flea Market","out"),("Spice plantation tour","out")]),
 "Kochi":     dict(country="India", lat=9.93,  lon=76.27, daily_usd=50,  tags=["culture","nature","food","relaxation"],
                   sights=[("Chinese fishing nets","out"),("Mattancherry Palace","in"),("Fort Kochi walk","out"),("Kerala Kathakali show","in"),("Backwater cruise","out")]),
 "Manali":    dict(country="India", lat=32.24, lon=77.19, daily_usd=45,  tags=["mountains","nature","adventure"],
                   sights=[("Solang Valley","out"),("Hadimba Temple","in"),("Old Manali cafes","in"),("Beas River walk","out"),("Rohtang Pass","out")]),
 # ---- Japan
 "Tokyo":     dict(country="Japan", lat=35.68, lon=139.69, daily_usd=150, tags=["culture","food","shopping","nightlife"],
                   sights=[("Senso-ji Temple","out"),("teamLab Planets","in"),("Tsukiji Outer Market","out"),("Shibuya Crossing","out"),("Tokyo National Museum","in")]),
 "Kyoto":     dict(country="Japan", lat=35.01, lon=135.77, daily_usd=130, tags=["culture","history","nature","food"],
                   sights=[("Fushimi Inari Shrine","out"),("Kinkaku-ji","out"),("Nishiki Market","in"),("Arashiyama Bamboo Grove","out"),("Kyoto International Manga Museum","in")]),
 "Osaka":     dict(country="Japan", lat=34.69, lon=135.50, daily_usd=120, tags=["food","nightlife","shopping","culture"],
                   sights=[("Osaka Castle","out"),("Dotonbori street food","out"),("Kaiyukan Aquarium","in"),("Kuromon Market","in"),("Universal Studios Japan","out")]),
 "Sapporo":   dict(country="Japan", lat=43.06, lon=141.35, daily_usd=110, tags=["food","nature","mountains"],
                   sights=[("Odori Park","out"),("Sapporo Beer Museum","in"),("Nijo Market","in"),("Mount Moiwa ropeway","out"),("Hokkaido Shrine","out")]),
 # ---- Italy
 "Rome":      dict(country="Italy", lat=41.90, lon=12.50, daily_usd=170, tags=["history","culture","food"],
                   sights=[("Colosseum","out"),("Vatican Museums","in"),("Trevi Fountain","out"),("Pantheon","in"),("Trastevere dinner","out")]),
 "Florence":  dict(country="Italy", lat=43.77, lon=11.26, daily_usd=160, tags=["culture","history","food"],
                   sights=[("Uffizi Gallery","in"),("Duomo and Brunelleschi's Dome","in"),("Ponte Vecchio","out"),("Boboli Gardens","out"),("Piazzale Michelangelo sunset","out")]),
 "Venice":    dict(country="Italy", lat=45.44, lon=12.32, daily_usd=190, tags=["culture","relaxation","history"],
                   sights=[("St Mark's Basilica","in"),("Gondola ride","out"),("Doge's Palace","in"),("Rialto Market","out"),("Burano island trip","out")]),
 "Naples":    dict(country="Italy", lat=40.85, lon=14.27, daily_usd=120, tags=["food","history","beach"],
                   sights=[("Pompeii","out"),("National Archaeological Museum","in"),("Pizza tasting in the historic centre","in"),("Mount Vesuvius hike","out"),("Spaccanapoli walk","out")]),
 # ---- Thailand
 "Bangkok":   dict(country="Thailand", lat=13.75, lon=100.50, daily_usd=70, tags=["food","culture","shopping","nightlife"],
                   sights=[("Grand Palace","out"),("Wat Pho","out"),("Jim Thompson House","in"),("Chatuchak Market","out"),("Asiatique night market","out")]),
 "Chiang Mai":dict(country="Thailand", lat=18.79, lon=98.98, daily_usd=55, tags=["culture","nature","mountains","food"],
                   sights=[("Doi Suthep Temple","out"),("Old City temples","out"),("Elephant sanctuary visit","out"),("Chiang Mai Night Bazaar","out"),("Thai cooking class","in")]),
 "Phuket":    dict(country="Thailand", lat=7.88,  lon=98.39, daily_usd=85, tags=["beach","nightlife","relaxation"],
                   sights=[("Patong Beach","out"),("Big Buddha","out"),("Old Phuket Town","out"),("Thai spa session","in"),("Phi Phi islands boat tour","out")]),
 "Krabi":     dict(country="Thailand", lat=8.09,  lon=98.91, daily_usd=75, tags=["beach","nature","adventure","relaxation"],
                   sights=[("Railay Beach","out"),("Four Islands tour","out"),("Tiger Cave Temple","out"),("Emerald Pool","out"),("Krabi Town night market","out")]),
}
print(len(CITIES), "cities in", sorted({d["country"] for d in CITIES.values()}))


def get_forecast(lat: float, lon: float, start_in_days: int, days: int) -> list:
    """Live daily forecast for the trip window. Returns a list of dicts:
    {"date": "2026-11-15", "tmax": 24.1, "tmin": 17.3, "rain_prob": 40}"""
    total = min(16, start_in_days + days)
    r = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": lat, "longitude": lon, "timezone": "auto", "forecast_days": total,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}, timeout=15).json()["daily"]
    rows = [{"date": r["time"][i], "tmax": r["temperature_2m_max"][i], "tmin": r["temperature_2m_min"][i],
             "rain_prob": r["precipitation_probability_max"][i] or 0} for i in range(len(r["time"]))]
    return rows[start_in_days:start_in_days + days]


def get_fx(base: str, target: str) -> Optional[float]:
    """Live exchange rate (1 base = ? target). Returns None if unavailable."""
    try:
        r = requests.get("https://api.frankfurter.dev/v1/latest", params={"base": base, "symbols": target}, timeout=10).json()
        return r["rates"][target]
    except Exception:
        return None

print(get_forecast(CITIES["Tokyo"]["lat"], CITIES["Tokyo"]["lon"], 1, 2))
print("1 USD in INR:", get_fx("USD", "INR"))

## 2. Given: shared state

All agents read and write one shared dictionary. Study it — the fields tell you what each node must return.

In [ ]:
class TripState(TypedDict, total=False):
    request: dict        # country, days, start_in_days, budget_per_day_usd, interests, home_currency
    candidates: list     # all dataset cities in the requested country        (scout)
    shortlist: list      # up to 3 chosen city names                          (scout)
    weather: dict        # city -> {"days": [...], "score": float}            (weather_analyst)
    budget_info: dict    # city -> {"daily_usd", "total_usd", "within_budget", "total_home"}  (budget_analyst)
    ranking: list        # [(city, score), ...] best first                    (decider)
    best_city: str       #                                                    (decider)
    over_budget: bool    # True if NO shortlisted city fits the budget        (decider)
    itinerary: str       #                                                    (planner)

REQUEST = {"country": "Japan", "days": 3, "start_in_days": 2, "budget_per_day_usd": 140,
           "interests": ["culture", "food"], "home_currency": "INR"}

## Task 1 — `score_weather` (10 marks)

Convert a forecast (list of daily dicts) into a **comfort score from 0 to 100** (higher = nicer trip weather).

Suggested rule (you may tune it, but the checks below must pass):
* Average temperature of a day = `(tmax + tmin) / 2`. Comfortable range is **18 – 28 °C**: no penalty inside it; outside it, penalty = `min(60, 5 × degrees outside the range)`.
* Rain penalty = `0.4 × rain_prob` (0–40 points).
* Day score = `100 − temperature penalty − rain penalty`. Return the **average day score**, rounded to 1 decimal.

💡 *Hint: write it as a pure function first and test with hand-made dicts — no LLM, no network.*

In [ ]:
def score_weather(days: list) -> float:
    """days: [{"tmax":..,"tmin":..,"rain_prob":..}, ...] -> float 0..100"""
    # TODO (Task 1)
    raise NotImplementedError("Task 1")

In [ ]:
def check(name, cond, msg=""):
    print(("✅ " if cond else "❌ ") + name + ("" if cond else f"  -> {msg}"))

def run_check(fn):
    try:
        fn()
    except NotImplementedError as e:
        print(f"⏳ not implemented yet: {e}")
    except Exception as e:
        print(f"❌ crashed: {type(e).__name__}: {e}")

def check_task1():
    nice = [{"tmax": 24, "tmin": 18, "rain_prob": 0}]
    awful = [{"tmax": 38, "tmin": 30, "rain_prob": 90}]
    cold_dry = [{"tmax": 2, "tmin": -4, "rain_prob": 0}]
    check("returns a number between 0 and 100", 0 <= score_weather(nice) <= 100)
    check("nice weather scores >= 90", score_weather(nice) >= 90, score_weather(nice))
    check("hot + rainy scores < 40", score_weather(awful) < 40, score_weather(awful))
    check("cold but dry is worse than nice", score_weather(cold_dry) < score_weather(nice))
    check("averages across days", abs(score_weather(nice + awful) - (score_weather(nice) + score_weather(awful)) / 2) < 0.2)
run_check(check_task1)

## 3. Given (worked example): `weather_analyst` node

A LangGraph node is just `function(state) -> dict of state updates`. This one loops over the shortlist, calls the live forecast tool and uses **your** `score_weather`. Use it as a model for the next tasks.

In [ ]:
def weather_node(state: TripState):
    req, out = state["request"], {}
    for city in state["shortlist"]:
        c = CITIES[city]
        days = get_forecast(c["lat"], c["lon"], req["start_in_days"], req["days"])
        out[city] = {"days": days, "score": score_weather(days)}
        print(f"[weather_analyst] {city}: score {out[city]['score']}")
    return {"weather": out}

## Task 2 — `scout_node` (15 marks): an LLM agent with structured output

1. Build `cands = {city: tags}` for dataset cities whose `country` matches `request["country"]` (case-insensitive).
2. Ask the LLM — using `llm.with_structured_output(Shortlist)` — to choose up to **3** cities best matching `request["interests"]`. Tell it to use **only** names from the list.
3. **Validate**: keep only names that are really in `cands` (small models make up names!). If fewer than 3 remain, **pad** with other candidates so we always compare 3 cities (or fewer if the country has fewer).
4. Return `{"candidates": [...], "shortlist": [...]}`.

💡 *Hints: `[SystemMessage(...), HumanMessage(...)]` is the message list; put the candidates and their tags in the human message.*

In [ ]:
class Shortlist(BaseModel):
    cities: list[str]

def scout_node(state: TripState):
    # TODO (Task 2)
    raise NotImplementedError("Task 2")

In [ ]:
def check_task2():
    out = scout_node({"request": REQUEST})
    check("returns 'candidates' and 'shortlist'", {"candidates", "shortlist"} <= set(out), list(out))
    check("shortlist has 1-3 cities", 1 <= len(out["shortlist"]) <= 3, out["shortlist"])
    check("all shortlisted cities are real Japanese cities", all(CITIES.get(c, {}).get("country") == "Japan" for c in out["shortlist"]), out["shortlist"])
    check("no duplicates", len(set(out["shortlist"])) == len(out["shortlist"]))
    other = scout_node({"request": {**REQUEST, "country": "thailand", "interests": ["beach"]}})
    check("country match is case-insensitive", all(CITIES[c]["country"] == "Thailand" for c in other["shortlist"]), other["shortlist"])
run_check(check_task2)

## Task 3 — `budget_node` (15 marks): cost analysis with a live FX tool

For every city in `state["shortlist"]` compute and return `{"budget_info": {city: {...}}}` where each entry has:

| key | meaning |
|---|---|
| `daily_usd` | the city's `daily_usd` |
| `total_usd` | `daily_usd × request["days"]` |
| `within_budget` | `daily_usd <= request["budget_per_day_usd"]` |
| `total_home` | `total_usd` converted to `request["home_currency"]` using `get_fx("USD", ...)`, rounded to a whole number; `None` if the rate is unavailable |

💡 *Call `get_fx` once, not once per city. Edge case: if the home currency is already `"USD"` the rate is 1 (the FX service rejects USD→USD).*

In [ ]:
def budget_node(state: TripState):
    # TODO (Task 3)
    raise NotImplementedError("Task 3")

In [ ]:
def check_task3():
    out = budget_node({"request": REQUEST, "shortlist": ["Tokyo", "Kyoto"]})["budget_info"]
    check("has an entry per shortlisted city", set(out) == {"Tokyo", "Kyoto"}, list(out))
    check("total_usd = daily x days", out["Tokyo"]["total_usd"] == 450 and out["Kyoto"]["total_usd"] == 390, out)
    check("Tokyo ($150/day) is over a $140 budget", out["Tokyo"]["within_budget"] is False)
    check("Kyoto ($130/day) is within budget", out["Kyoto"]["within_budget"] is True)
    check("total_home is a converted number", isinstance(out["Kyoto"]["total_home"], int) and out["Kyoto"]["total_home"] > 390, out["Kyoto"]["total_home"])
    usd = budget_node({"request": {**REQUEST, "home_currency": "USD"}, "shortlist": ["Kyoto"]})["budget_info"]["Kyoto"]
    check("home currency USD -> total_home == total_usd", usd["total_home"] == 390, usd["total_home"])
run_check(check_task3)

## Task 4 — `decider_node` (15 marks): combine the evidence (plain code, no LLM)

Not every node needs an LLM! Deterministic logic is cheaper, faster and 100 % reliable.

1. `pool` = shortlisted cities that are **within budget**. If the pool is empty, use `[cheapest shortlisted city]` and set `over_budget = True`.
2. For each city in the pool: `affordability = 100 × cheapest_daily_in_pool / city_daily` (cheapest city gets 100).
3. `final = 0.6 × weather_score + 0.4 × affordability` (round to 1 decimal).
4. Return `{"ranking": [(city, final), ...] sorted best-first, "best_city": ..., "over_budget": bool}`.

In [ ]:
def decider_node(state: TripState):
    # TODO (Task 4)
    raise NotImplementedError("Task 4")

In [ ]:
def check_task4():
    fake = {
        "shortlist": ["A", "B", "C"],
        "weather": {"A": {"score": 90}, "B": {"score": 60}, "C": {"score": 95}},
        "budget_info": {"A": {"daily_usd": 100, "within_budget": True},
                        "B": {"daily_usd": 50,  "within_budget": True},
                        "C": {"daily_usd": 300, "within_budget": False}},
    }
    out = decider_node(fake)
    check("over-budget city C excluded", all(c != "C" for c, _ in out["ranking"]), out["ranking"])
    check("ranking is sorted best-first", out["ranking"][0][1] >= out["ranking"][-1][1])
    check("B wins with these weights (B=76.0 vs A=74.0)", out["best_city"] == "B", out["ranking"])
    check("over_budget False when something fits", out["over_budget"] is False)
    none_fit = {**fake, "budget_info": {k: {**v, "within_budget": False} for k, v in fake["budget_info"].items()}}
    out2 = decider_node(none_fit)
    check("if nothing fits: pick cheapest and flag over_budget", out2["best_city"] == "B" and out2["over_budget"] is True, out2)
run_check(check_task4)

## Task 5 — `planner_node` (15 marks): a grounded itinerary writer (LLM)

Write the itinerary for `state["best_city"]`. **Ground** the LLM — give it *only* the facts it may use, otherwise a small model will invent attractions:

* the city's `sights` (name + indoor/outdoor) from `CITIES`,
* each trip day's date, temperature range and rain probability from `state["weather"][city]["days"]`.

Your prompt must instruct the model to:
* output **exactly one block per day**, labelled `Day 1`, `Day 2`, … with *Morning / Afternoon / Evening*;
* use **only** the listed sights (you may add generic meals/rest);
* put **indoor** sights on days with `rain_prob >= 50` and prefer outdoor sights on dry days.

Return `{"itinerary": text}`.

💡 *Hint: build a readable "facts" string first (one line per day, one line per sight), then pass it to the LLM. Temperature is 0 so results are repeatable.*

In [ ]:
def planner_node(state: TripState):
    # TODO (Task 5)
    raise NotImplementedError("Task 5")

In [ ]:
def check_task5():
    city = "Kyoto"
    fake = {"request": REQUEST, "best_city": city,
            "weather": {city: {"days": [
                {"date": "2026-11-15", "tmin": 10, "tmax": 18, "rain_prob": 10},
                {"date": "2026-11-16", "tmin": 9,  "tmax": 15, "rain_prob": 90},
                {"date": "2026-11-17", "tmin": 10, "tmax": 19, "rain_prob": 0}]}}}
    text = planner_node(fake)["itinerary"]
    names = [n for n, _ in CITIES[city]["sights"]]
    check("returns text", isinstance(text, str) and len(text) > 100)
    check("has Day 1, Day 2 and Day 3", all(f"Day {i}" in text for i in (1, 2, 3)))
    check("mentions at least 3 real sights", sum(n in text for n in names) >= 3, [n for n in names if n in text])
    d2 = text.split("Day 2")[-1].split("Day 3")[0]
    indoor = [n for n, k in CITIES[city]["sights"] if k == "in"]
    check("rainy Day 2 uses an indoor sight (LLM-dependent - re-run once if flaky)", any(n in d2 for n in indoor), d2[:200])
    print("\n--- sample output ---\n" + text)
run_check(check_task5)

## Task 6 — Wire the graph and run end-to-end (15 marks)

Build the `StateGraph` shown at the top:

* add the 5 nodes with names: `scout`, `weather_analyst`, `budget_analyst`, `decider`, `planner` (nodes: `scout_node`, `weather_node`, `budget_node`, `decider_node`, `planner_node`);
* `START → scout`;
* **fan-out:** `scout → weather_analyst` **and** `scout → budget_analyst` (they run in parallel);
* **fan-in:** both → `decider` (it runs only after *both* finish);
* `decider → planner → END`;
* compile into `travel_planner`.

💡 *Parallel branches write different state keys (`weather` vs `budget_info`), so no reducer is needed.*

In [ ]:
g = StateGraph(TripState)
# TODO (Task 6): add nodes and edges

travel_planner = None   # TODO: g.compile()
travel_planner

In [ ]:
def show_report(state):
    b = state["best_city"]; bi = state["budget_info"][b]
    print("=" * 70)
    print(f"BEST CITY: {b}   (ranking: {state['ranking']})")
    if state["over_budget"]:
        print("⚠️  No shortlisted city fits your daily budget - showing the cheapest option.")
    print(f"Estimated cost: ${bi['total_usd']} total  (~{bi['total_home']} {REQUEST['home_currency']})")
    print("=" * 70)
    print(state["itinerary"])

def check_task6():
    check("graph compiled", travel_planner is not None, "compile the graph")
    result = travel_planner.invoke({"request": REQUEST})
    need = {"shortlist", "weather", "budget_info", "ranking", "best_city", "over_budget", "itinerary"}
    check("final state has every field", need <= set(result), need - set(result))
    check("best city is in Japan", CITIES[result["best_city"]]["country"] == "Japan")
    check("itinerary exists", len(result["itinerary"]) > 100)
    show_report(result)
run_check(check_task6)

Try other requests once it works. Compare how the choice changes with budget:

In [ ]:
def plan(**changes):
    req = {**REQUEST, **changes}
    result = travel_planner.invoke({"request": req})
    globals()["REQUEST"] = req          # so show_report prints the right currency
    show_report(result)

plan(country="Thailand", interests=["beach", "relaxation"], budget_per_day_usd=80, days=4, home_currency="USD")
plan(country="Italy", budget_per_day_usd=100)      # nothing fits -> over_budget path

## Task 7 — Reflection (15 marks)

Answer in 2–4 sentences each, in your own words (edit this cell).

1. Why did we make `decider` and `budget_analyst` plain Python instead of LLM agents? When *would* an LLM be the better choice?
2. What went wrong (or could go wrong) when the LLM invents a city or a sight? Which lines of **your** code protect against that?
3. `weather_analyst` and `budget_analyst` run in parallel. Why is that safe here? What would break if both wrote to the same state key without a reducer?
4. Run the same request with a different model (e.g. `qwen2.5`). What changed in the scout's shortlist and the itinerary quality?
5. Name one real-world limitation of this planner (data, forecasts, costs, …) and how you would fix it.

**Your answers:**

1.
2.
3.
4.
5.

## ★ Stretch goals (+10 bonus, pick any — 2–5 marks each)

* **A. Reviewer agent (loop):** add a `reviewer` node after `planner` that checks (with code and/or an LLM) that every day is present, only real sights are used, and rainy days are indoor. If it fails, route back to `planner` with feedback via `add_conditional_edges` (max 2 retries).
* **B. Human in the loop:** compile with `MemorySaver` and `interrupt_before=["planner"]` so the user can confirm the chosen city before the itinerary is written.
* **C. Trips further than 16 days away:** swap `get_forecast` for Open-Meteo's climate/historical API (`archive-api.open-meteo.com`) to use last year's weather for the same dates.
* **D. Free-text input:** add a `parse_request` node using structured output so users can type *"4 days in Italy in a week, around $150 a day, I love food and history"*.
* **E. Per-agent models:** use a small model for `scout` and a larger one for `planner`; report quality/time differences.
* **F. Simple UI:** wrap `travel_planner.invoke` in a Gradio or Streamlit form.

## Submission
Submit this notebook **with all outputs visible** (Kernel → Restart & Run All first). Include your name and the Ollama model you used in the cell below.

In [ ]:
STUDENT_NAME = ""     # TODO
MODEL_USED = MODEL
print(STUDENT_NAME, MODEL_USED)